In [10]:
import argparse
import os
import random
from collections import OrderedDict
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import tqdm
from scipy.stats import pearsonr, spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision.utils import save_image

from albumentations.augmentations import transforms
from albumentations.core.composition import Compose, OneOf
from sklearn.model_selection import train_test_split

from easydict import EasyDict as edict
from torch.optim import lr_scheduler

from archs import gigatime
from losses import *
from utils import *
from prov_data import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


Argument Parser

In [11]:
import argparse

def str2bool(v):
    return v.lower() in ('true', '1')

cwd = Path.cwd().resolve()

metadata_candidates = [
    "/data/full_metadata.csv",
    "/data/sample_test_data/sample_metadata.csv",
    str((cwd / "data" / "full_metadata.csv").resolve()),
    str((cwd / "data" / "sample_test_data" / "sample_metadata.csv").resolve()),
    str((cwd / ".." / "data" / "full_metadata.csv").resolve()),
    str((cwd / ".." / "data" / "sample_test_data" / "sample_metadata.csv").resolve()),
]
default_metadata = next((p for p in metadata_candidates if Path(p).is_file()), metadata_candidates[0])

tiling_candidates = [
    "/data/gigatime_training_tiles/",
    "/data/sample_test_data/data/",
    str((cwd / "data" / "gigatime_training_tiles").resolve()),
    str((cwd / "data" / "sample_test_data" / "data").resolve()),
    str((cwd / ".." / "data" / "gigatime_training_tiles").resolve()),
    str((cwd / ".." / "data" / "sample_test_data" / "data").resolve()),
]
default_tiling_dir = next((p for p in tiling_candidates if Path(p).is_dir()), tiling_candidates[0])

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument('--name', default="gigatime_training")
    parser.add_argument('--output_dir', default="./scratch")
    parser.add_argument('--gpu_ids', nargs='+', type=int)
    parser.add_argument('--metadata', default=default_metadata)
    parser.add_argument('--tiling_dir', default=default_tiling_dir)
    parser.add_argument('--epochs', default=1, type=int)
    parser.add_argument('--batch_size', default=32, type=int)

    # model
    parser.add_argument('--arch', default='NestedUNet')
    parser.add_argument('--input_channels', default=3, type=int)
    parser.add_argument('--num_classes', default=23, type=int)
    parser.add_argument('--input_w', default=512, type=int)
    parser.add_argument('--input_h', default=512, type=int)

    # loss
    parser.add_argument('--loss', default='BCEDiceLoss')

    # optimizer
    parser.add_argument('--optimizer', default='Adam', choices=['Adam', 'SGD'])
    parser.add_argument('--lr', default=1e-3, type=float)
    parser.add_argument('--momentum', default=0.9, type=float)
    parser.add_argument('--weight_decay', default=1e-4, type=float)
    parser.add_argument('--nesterov', default=False, type=str2bool)

    # scheduler
    parser.add_argument('--scheduler', default='CosineAnnealingLR',
                        choices=['CosineAnnealingLR', 'ReduceLROnPlateau', 'MultiStepLR', 'ConstantLR'])
    parser.add_argument('--min_lr', default=1e-5, type=float)
    parser.add_argument('--factor', default=0.1, type=float)
    parser.add_argument('--patience', default=2, type=int)
    parser.add_argument('--milestones', default='1,2', type=str)
    parser.add_argument('--gamma', default=2/3, type=float)
    parser.add_argument('--early_stopping', default=-1, type=int)

    parser.add_argument('--num_workers', default=12, type=int)
    parser.add_argument('--window_size', type=int, default=256)
    parser.add_argument('--sampling_prob', type=float, default=0.5)
    parser.add_argument('--val_sampling_prob', type=float, default=0.01)
    parser.add_argument('--transformer', type=str2bool, default=False)
    parser.add_argument('--sigmoid', type=str2bool, default=True)
    parser.add_argument('--crop', type=str2bool, default=False)

    return edict(vars(parser.parse_args([])))

config = parse_args()
print(f"Using metadata: {config['metadata']}")
print(f"Using tiling_dir: {config['tiling_dir']}")

Using metadata: /Users/robertkramer/providence/prog_prov/GigaTIME/data/sample_test_data/sample_metadata.csv
Using tiling_dir: /Users/robertkramer/providence/prog_prov/GigaTIME/data/sample_test_data/data


Metrics

In [12]:
mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
std = torch.tensor([0.229, 0.224, 0.225]).to(device)


def calculate_correlations(matrix1, matrix2):
    """
    Calculate Pearson and Spearman correlation coefficients between two matrices.

    Args:
        matrix1 (np.ndarray): The first matrix.
        matrix2 (np.ndarray): The second matrix.

    Returns:
        dict: A dictionary containing Pearson and Spearman correlation coefficients.
    """
    assert matrix1.shape == matrix2.shape, "Matrices must have the same shape"
    b, c, h, w = matrix1.shape

    pearson_correlations = []
    spearman_correlations = []

    for channel in range(c):
        pearson_corrs = []
        spearman_corrs = []

        for batch in range(b):
            flat_matrix1 = matrix1[batch, channel].flatten()
            flat_matrix2 = matrix2[batch, channel].flatten()

            valid_indices = ~np.isnan(flat_matrix1.cpu().numpy()) & ~np.isnan(flat_matrix2.cpu().numpy())
            flat_matrix1 = flat_matrix1[valid_indices]
            flat_matrix2 = flat_matrix2[valid_indices]

            if len(flat_matrix1) > 0 and len(flat_matrix2) > 0:
                pearson_corr, _ = pearsonr(flat_matrix1.cpu().numpy(), flat_matrix2.cpu().numpy())
                spearman_corr, _ = spearmanr(flat_matrix1.cpu().numpy(), flat_matrix2.cpu().numpy())
            else:
                pearson_corr = np.nan
                spearman_corr = np.nan

            pearson_corrs.append(pearson_corr)
            spearman_corrs.append(spearman_corr)

        pearson_correlations.append(np.nanmean(pearson_corrs))
        spearman_correlations.append(np.nanmean(spearman_corrs))

    return pearson_correlations, spearman_correlations

def split_into_boxes(tensor, box_size):
    batch_size, channels, height, width = tensor.shape
    num_boxes_y = height // box_size
    num_boxes_x = width // box_size

    boxes = tensor.unfold(2, box_size, box_size).unfold(3, box_size, box_size)
    boxes = boxes.contiguous().view(batch_size, channels, num_boxes_y, num_boxes_x, box_size, box_size)

    return boxes

def count_ones(boxes):
    return boxes.sum(dim=(4, 5))

def get_box_metrics(pred, mask, box_size):
    pred_boxes = split_into_boxes(pred, box_size)
    mask_boxes = split_into_boxes(mask, box_size)
    pred_counts = count_ones(pred_boxes)
    mask_counts = count_ones(mask_boxes)

    mse = ((pred_counts.float() - mask_counts.float()) ** 2).mean(dim=0)
    mean_mse_per_channel = mse.mean(dim=(1,2))
    mean_mse = mse.mean().item()

    pearson, spearman = calculate_correlations(pred_counts, mask_counts)

    return mean_mse_per_channel, pearson, spearman

Data Loader Helpers

In [13]:
def sample_data_loader(data_loader, config, sample_fraction=0.1, deterministic=False, what_split="train"):
    # this just samples some fraction of the data in the dataloader so that we can train on a smaller subset for quick testing
    dataset = data_loader.dataset
    total_size = len(dataset)
    sample_size = int(total_size * sample_fraction)

    if deterministic:
        sample_indices = [i for i in range(sample_size)]
    else:
        sample_indices = random.sample(range(total_size), sample_size)

    subset = Subset(dataset, sample_indices)

    if what_split == "train":
        sample_loader = DataLoader(subset, batch_size=data_loader.batch_size, shuffle=True,
            num_workers=config['num_workers'], prefetch_factor=6, drop_last=True)
    else:
        sample_loader = DataLoader(subset, batch_size=data_loader.batch_size, shuffle=False,
            num_workers=config['num_workers'], prefetch_factor=6, drop_last=False)
    return sample_loader

Training & Validation Loops

In [14]:
def train(config, train_loader, model, criterion, optimizer):
    avg_meters = {'loss': AverageMeter(), 'pearson': AverageMeter()}
    pearson_per_class_meters = [AverageMeter() for _ in range(config['num_classes'])]
    window_size = config['window_size']
    
    model.train()

    pbar = tqdm.tqdm(total=len(train_loader))
    for input, target, name in train_loader:
        downsampled_image = F.interpolate(target, scale_factor=1/8, mode='bilinear', align_corners=False)
        target = F.interpolate(downsampled_image, size=(config["input_h"],config["input_h"]), mode='bilinear', align_corners=False)
        target = target.to(device)
        
        output_image = model(input.to(device)).to(device)

        loss = criterion(output_image, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, pearson, _ = get_box_metrics(output_image, target, box_size=8)

        for class_idx, pearson_value in enumerate(pearson):
            pearson_per_class_meters[class_idx].update(pearson_value, input.size(0))

        avg_meters['loss'].update(loss.item(), input.size(0))
        avg_meters['pearson'].update(np.nanmean(pearson), input.size(0))

        pbar.set_postfix({'loss': avg_meters['loss'].avg, 'pearson': avg_meters['pearson'].avg})
        pbar.update(1)
    pbar.close()

    return OrderedDict([('loss', avg_meters['loss'].avg), ('pearson', avg_meters['pearson'].avg)] +
                       [(f'class_{i}', m.avg) for i, m in enumerate(pearson_per_class_meters)])

def validate(config, val_loader, model, criterion):
    avg_meters = {'loss': AverageMeter(), 'pearson': AverageMeter()}
    pearson_per_class_meters = [AverageMeter() for _ in range(config['num_classes'])]
    window_size = config['window_size']
    
    model.eval()

    with torch.no_grad():
        pbar = tqdm.tqdm(total=len(val_loader))
        for input, target, name in val_loader:
            downsampled_image = F.interpolate(target, scale_factor=1/8, mode='bilinear', align_corners=False)
            target = F.interpolate(downsampled_image, size=(config["input_h"],config["input_h"]), mode='bilinear', align_corners=False)
            target = target.to(device)
            
            output_image = model(input.to(device)).to(device)

            loss = criterion(output_image, target)
            
            _, pearson, _ = get_box_metrics(output_image, target, box_size=8)
            
            for class_idx, pearson_value in enumerate(pearson):
                pearson_per_class_meters[class_idx].update(pearson_value, input.size(0))

            avg_meters['loss'].update(loss.item(), input.size(0))
            avg_meters['pearson'].update(np.nanmean(pearson), input.size(0))

            pbar.set_postfix({'loss': avg_meters['loss'].avg, 'pearson': avg_meters['pearson'].avg})
            pbar.update(1)
        pbar.close()

    return OrderedDict([('loss', avg_meters['loss'].avg), ('pearson', avg_meters['pearson'].avg)] +
                       [(f'class_{i}', m.avg) for i, m in enumerate(pearson_per_class_meters)])

Model, Loss, Optimizer, Scheduler

In [15]:
common_channel_list=['DAPI', 
    'TRITC',
    'Cy5',
    'PD-1',
    'CD14',
    'CD4',
    'T-bet',
    'CD34',
    'CD68',
    'CD16',
    'CD11c',
    'CD138',
    'CD20',
    'CD3',
    'CD8',
    'PD-L1',
    'CK',
    'Ki67',
    'Tryptase',
    'Actin-D',
    'Caspase3-D',
    'PHH3-B',
    'Transgelin']

if config['loss'] == 'MSELoss':
    criterion = nn.MSELoss().to(device)
elif config['loss'] == 'BCEWithLogitsLoss':
    criterion = nn.BCEWithLogitsLoss().to(device)
elif config['loss'] == 'BCEDiceLoss':
    criterion = BCEDiceLoss().to(device)
else:
    criterion = losses.__dict__[config['loss']]().to(device)

model = gigatime(num_classes=config['num_classes'],
                 sigmoid=config["sigmoid"],
                 loss_type=config["loss"],
                 input_channels=config['input_channels']).to(device)

if torch.cuda.is_available() and config["gpu_ids"] and len(config["gpu_ids"]) > 1:
    model = nn.DataParallel(model, device_ids=config["gpu_ids"])
    print("using multiple GPUs", config["gpu_ids"])

params = filter(lambda p: p.requires_grad, model.parameters())
if config['optimizer'] == 'Adam':
    optimizer = optim.Adam(params, lr=config['lr'], weight_decay=config['weight_decay'])
elif config['optimizer'] == 'SGD':
    optimizer = optim.SGD(params, lr=config['lr'], momentum=config['momentum'],
                          nesterov=config['nesterov'], weight_decay=config['weight_decay'])

if config['scheduler'] == 'CosineAnnealingLR':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=config['min_lr'])
elif config['scheduler'] == 'ReduceLROnPlateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=config['factor'],
                                                     patience=config['patience'], verbose=1,
                                                     min_lr=config['min_lr'])
elif config['scheduler'] == 'MultiStepLR':
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer,
                                               milestones=[int(e) for e in config['milestones'].split(',')],
                                               gamma=config['gamma'])
elif config['scheduler'] == 'ConstantLR':
    scheduler = None

Dataset Creation

In [16]:
data_ready = True
tile_pair_df = None
tile_pair_df_filtered_dicefilter = None
dataset_dir_root = config["tiling_dir"]

metadata_path = Path(config["metadata"]).resolve()
tiling_dir_path = Path(config["tiling_dir"]).resolve()

if not tiling_dir_path.is_dir():
    data_ready = False
    print(f"Tiling directory not found at {tiling_dir_path}; dataset/training cells will be skipped.")

if data_ready:
    try:
        metadata = pd.read_csv(metadata_path) if metadata_path.is_file() else None
        has_full_metadata = metadata is not None and {"tiff_filename", "he_filename"}.issubset(metadata.columns)
        has_img_stats = any(tiling_dir_path.rglob("img_statistics.json"))

        if has_full_metadata and has_img_stats:
            # Full training-data mode
            tile_pair_df = generate_tile_pair_df(metadata=metadata, tiling_dir=tiling_dir_path)
            tile_pair_df_filtered = tile_pair_df[tile_pair_df.apply(
                lambda x: (x["img_comet_black_ratio"] < 0.3) &
                          (x["img_comet_variance"] > 200) &
                          (x["img_he_black_ratio"] < 0.3) &
                          (x["img_he_variance"] > 200), axis=1
)]

            # Attach dice metrics (prefer segment_metric.json, fallback to per-tile *_dice_metric.json)
            dir_names = tile_pair_df_filtered["dir_name"].unique()
            dice_values = []
            for _, row in tile_pair_df_filtered.iterrows():
                pair_name = row["pair_name"]
                dir_name = Path(row["dir_name"])
                segment_path = dir_name / "segment_metric.json"
                dice_path = dir_name / f"{pair_name}_dice_metric.json"
                dice_val = 1.0

                if segment_path.is_file():
                    with open(segment_path, "r") as f:
                        segment_metric_dict = json.load(f)
                    dice_val = segment_metric_dict.get(pair_name, {}).get("dice", 1.0)
                elif dice_path.is_file():
                    with open(dice_path, "r") as f:
                        dice_payload = json.load(f)
                    if isinstance(dice_payload, dict):
                        if "dice" in dice_payload and isinstance(dice_payload["dice"], (int, float)):
                            dice_val = float(dice_payload["dice"])
                        else:
                            numeric_vals = [float(v) for v in dice_payload.values() if isinstance(v, (int, float))]
                            if numeric_vals:
                                dice_val = float(np.mean(numeric_vals))

                dice_values.append(dice_val)

            tile_pair_df_filtered = tile_pair_df_filtered.copy()
            tile_pair_df_filtered["dice"] = dice_values
            tile_pair_df_filtered_dicefilter = tile_pair_df_filtered[tile_pair_df_filtered["dice"] > 0.2]
            dataset_dir_root = str(tiling_dir_path)
            print(f"Using full metadata mode with {len(tile_pair_df_filtered_dicefilter)} filtered tiles.")

        else:
            # Fallback mode for sample/alternate folder layouts
            he_paths = sorted(tiling_dir_path.rglob("*_he.png"))
            if not he_paths:
                raise FileNotFoundError(
                    f"No '*_he.png' files found under {tiling_dir_path}."
                )

            pair_names = [p.name.replace("_he.png", "") for p in he_paths]
            parent_dirs = [str(p.parent) for p in he_paths]

            tile_pair_df = pd.DataFrame({
                "pair_name": pair_names,
                "dir_name": parent_dirs,
                "dice": 1.0,
            })
            tile_pair_df_filtered_dicefilter = tile_pair_df.copy()

            common_parent = Path(os.path.commonpath(parent_dirs))
            dataset_dir_root = str(common_parent.parent if common_parent.name == "data" else common_parent)

            print(f"Using fallback tile-discovery mode with {len(tile_pair_df)} tiles.")
            print(f"Dataset dir root for loader: {dataset_dir_root}")

    except Exception as exc:
        data_ready = False
        print(f"Dataset preparation skipped due to: {exc}")

Using fallback tile-discovery mode with 50 tiles.
Dataset dir root for loader: /Users/robertkramer/providence/prog_prov/GigaTIME/data/sample_test_data


Data loader setup

In [19]:
if data_ready:
    import albumentations as geometric

    loader_dir_path = dataset_dir_root if dataset_dir_root else config["tiling_dir"]
    unique_dirs = tile_pair_df_filtered_dicefilter["dir_name"].nunique()
    use_full_split = unique_dirs < 2
    train_split = "full" if use_full_split else "train"
    val_split = "full" if use_full_split else "valid"
    val_standard = "all" if use_full_split else "silver"

    print(f"Using loader dir_path: {loader_dir_path}")
    if use_full_split:
        print("Only one directory group detected; using full split mode for both train/val.")

    if config['crop']:
        train_transform = Compose([
            geometric.RandomRotate90(),
            geometric.HorizontalFlip(p=0.5),
            OneOf([
                transforms.HueSaturationValue(),
                transforms.RandomBrightnessContrast(brightness_limit=0, contrast_limit=0.2),
                transforms.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0),
            ], p=1),
            geometric.RandomCrop(config['input_h'], config['input_w']),
            transforms.Normalize()
        ],
            is_check_shapes=False)

        val_transform = Compose([
            geometric.Resize(config['input_h'], config['input_w']),
            transforms.Normalize()
        ],
            is_check_shapes=False)

    else:
        train_transform = Compose([
            geometric.RandomRotate90(),
            geometric.HorizontalFlip(p=0.5),
            OneOf([
                transforms.HueSaturationValue(),
                transforms.RandomBrightnessContrast(brightness_limit=0, contrast_limit=0.2),
                transforms.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0),
            ], p=1),
            geometric.Resize(config['input_h'], config['input_w']),
            transforms.Normalize()
        ],
            is_check_shapes=False)

        val_transform = Compose([
            geometric.Resize(config['input_h'], config['input_w']),
            transforms.Normalize()
        ],
            is_check_shapes=False)

    train_dataset = HECOMETDataset_roi(
            all_tile_pair=tile_pair_df,
            tile_pair_df=tile_pair_df_filtered_dicefilter,
            transform=train_transform,
            dir_path=loader_dir_path,
            window_size=config["window_size"],
            split=train_split,
            mask_noncell=True,
            cell_mask_label=True,
        )

    val_dataset = HECOMETDataset_roi(
        all_tile_pair=tile_pair_df,
        tile_pair_df=tile_pair_df_filtered_dicefilter,
        transform=val_transform,
        dir_path=loader_dir_path,
        window_size=config["window_size"],
        split=val_split,
        standard=val_standard,
        mask_noncell=True,
        cell_mask_label=True,
    )

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,
                              num_workers=config['num_workers'], prefetch_factor=6, drop_last=True)

    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False,
                            num_workers=config['num_workers'], prefetch_factor=6, drop_last=False)

    if not use_full_split:
        val_loader = sample_data_loader(val_loader, config, config['val_sampling_prob'], deterministic=True, what_split="valid")
else:
    train_loader = None
    val_loader = None
    print("Skipping dataloader setup because required training dataset files are unavailable.")

Using loader dir_path: /Users/robertkramer/providence/prog_prov/GigaTIME/data/sample_test_data
Only one directory group detected; using full split mode for both train/val.


/Users/robertkramer/providence/prog_prov/GigaTIME/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Training loop

In [20]:
if data_ready and train_loader is not None and val_loader is not None:
    for epoch in range(config['epochs']):
        print(f"Epoch [{epoch+1}/{config['epochs']}]")

        # --- Train ---
        train_log = train(config, train_loader, model, criterion, optimizer)

        # --- Validate ---
        val_log = validate(config, val_loader, model, criterion)

        print(f"Train -> Loss: {train_log['loss']:.4f}, IoU: {train_log['pearson']:.4f}")
        print(f"Val   -> Loss: {val_log['loss']:.4f}, IoU: {val_log['pearson']:.4f}")

        print("End of Epoch 1")
        print("Change settings (epochs) to train fully or use the train.py script to train the model")
else:
    print("Skipping training loop because required training dataset files are unavailable.")

Epoch [1/1]


  0%|          | 0/1 [00:00<?, ?it/s]/Users/robertkramer/providence/prog_prov/GigaTIME/.venv/lib/python3.11/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/Users/robertkramer/providence/prog_prov/GigaTIME/scripts/prov_data.py:266: FutureWarning: `RegionProperties.convex_image` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.image_convex` instead. 
  region_mask = region.convex_image # Shape: (region_height, region_width)
/Users/robertkramer/providence/prog_prov/GigaTIME/.venv/lib/python3.11/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks

: 